In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

GPU available: True
Device: Tesla T4


##  LOAD DATA

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/output.csv")

df.head()

,News ID,Category,Topic,Headline,News body,Title entity,Entity content
0,N10000,sports,soccer,Predicting Atlanta United's lineup against Col...,"Only FIVE internationals allowed, count em, FI...","{""Atlanta United's"": 'Atlanta United FC'}","{'Atlanta United FC': {'type': 'item', 'id': '..."
1,N10001,news,newspolitics,Mitch McConnell: DC statehood push is 'full bo...,WASHINGTON -- Senate Majority Leader Mitch McC...,"{'DC': 'Washington, D.C.'}","{'Washington, D.C.': {'type': 'item', 'id': 'Q..."
2,N10002,news,newsus,Home In North Highlands Damaged By Fire,NORTH HIGHLANDS (CBS13) Fire damaged a home ...,{},{}
3,N10003,news,newspolitics,Meghan McCain blames 'liberal media' and 'thir...,Meghan McCain is speaking out after a journali...,{},{}
4,N10004,news,newsworld,Today in History: Aug 1,"1714: George I becomes King Georg Ludwig, Elec...",{},{}


## BALANCED SAMPLING (20K)

In [ ]:
TARGET_SIZE = 20000

def get_main_entity(row):
    text = str(row["Title entity"]) + " " + str(row["Entity content"])
    text = text.lower()

    if "person" in text:
        return "PER"
    elif "org" in text:
        return "ORG"
    elif "loc" in text:
        return "LOC"
    elif "date" in text:
        return "DATE"
    else:
        return "MISC"

df["main_label"] = df.apply(get_main_entity, axis=1)

samples_per_class = TARGET_SIZE // df["main_label"].nunique()

balanced_df = (
    df.groupby("main_label", group_keys=False)
      .apply(lambda x: x.sample(min(len(x), samples_per_class), random_state=42))
)

balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

df = balanced_df

print("✅ Balanced data:", df.shape)

✅ Balanced data: (12028, 8)


/tmp/ipykernel_30304/685112922.py:24: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), samples_per_class), random_state=42))


In [ ]:
df["text"] = df["Headline"] + ". " + df["News body"]

In [ ]:
print(df.columns)

Index(['News ID', 'Category', 'Topic', 'Headline', 'News body', 'Title entity',
       'Entity content', 'main_label', 'text'],
      dtype='object')


In [ ]:
import ast
import re
import spacy
from tqdm import tqdm

# ✅ Use small model for speed
nlp = spacy.load("en_core_web_sm")

# -----------------------------
# Normalize text
# -----------------------------
def normalize(text):
    return re.sub(r"\s+", " ", str(text).strip())

# -----------------------------
# Entity Type Inference
# -----------------------------
def infer_entity_type(name):
    name = str(name)
    name_lower = name.lower()

    if re.search(r"\b\d{4}\b", name):
        return "DATE"

    if any(x in name_lower for x in ["india", "usa", "uk", "washington"]):
        return "LOC"

    if any(k in name_lower for k in ["inc", "ltd", "corp", "company"]):
        return "ORG"

    if len(name.split()) >= 2 and name.replace(".", "").istitle():
        return "PER"

    return "MISC"

In [ ]:
# -----------------------------
# Extract entity dictionary
# -----------------------------
def extract_entities(title_ent_str, content_ent_str):
    entities = {}

    try:
        title_dict = ast.literal_eval(title_ent_str)
        entities.update(title_dict)
    except:
        pass

    try:
        content_dict = ast.literal_eval(content_ent_str)
        for k in content_dict:
            entities[k] = k
    except:
        pass

    return entities

In [ ]:
# -----------------------------
# Convert to BIO
# -----------------------------
def convert_to_bio(doc, title_ent_str, content_ent_str):
    tokens = [token.text for token in doc]
    tags = ["O"] * len(tokens)

    # 🔥 Step 1: spaCy NER
    for ent in doc.ents:
        mapped = {
            "PERSON": "PER",
            "GPE": "LOC",
            "LOC": "LOC",
            "ORG": "ORG",
            "DATE": "DATE"
        }.get(ent.label_, "MISC")

        tags[ent.start] = f"B-{mapped}"
        for i in range(ent.start + 1, ent.end):
            tags[i] = f"I-{mapped}"

    # 🔥 Step 2: Custom entities
    entities = extract_entities(title_ent_str, content_ent_str)

    token_spans = [(token.idx, token.idx + len(token.text)) for token in doc]

    for surface, expanded in sorted(entities.items(), key=lambda x: len(x[0]), reverse=True):

        ent_type = infer_entity_type(expanded)

        pattern = r'(?<!\w)' + re.escape(surface) + r'(?!\w)'

        for match in re.finditer(pattern, doc.text, re.IGNORECASE):
            start_char, end_char = match.start(), match.end()

            token_indices = [
                i for i, (s, e) in enumerate(token_spans)
                if not (e <= start_char or s >= end_char)
            ]

            if not token_indices:
                continue

            # ❗ don't overwrite spaCy labels
            if any(tags[i] != "O" for i in token_indices):
                continue

            tags[token_indices[0]] = f"B-{ent_type}"
            for idx in token_indices[1:]:
                tags[idx] = f"I-{ent_type}"

    return tokens, tags

In [ ]:
# -----------------------------
# 🚀 CHUNK PROCESSING
# -----------------------------
sentences = []
labels = []

CHUNK_SIZE = 5000

print("🚀 Processing in chunks...")

for start in range(0, len(df), CHUNK_SIZE):
    end = start + CHUNK_SIZE
    chunk = df.iloc[start:end]

    texts = chunk["text"].astype(str).apply(normalize).tolist()

    docs = nlp.pipe(texts, batch_size=64)

    for doc, (_, row) in tqdm(zip(docs, chunk.iterrows()), total=len(chunk)):
        tokens, tags = convert_to_bio(
            doc,
            row["Title entity"],
            row["Entity content"]
        )

        sentences.append(tokens)
        labels.append(tags)

    print(f"✅ Processed {start} to {end}")

print("🎉 DONE:", len(sentences))

# -----------------------------
# Preview
# -----------------------------
for i in range(3):
    print("\nSentence:", sentences[i])
    print("Tags:", labels[i])

🚀 Processing in chunks...


100%|██████████| 5000/5000 [09:52<00:00,  8.44it/s]


✅ Processed 0 to 5000


100%|██████████| 5000/5000 [09:16<00:00,  8.99it/s]


✅ Processed 5000 to 10000


100%|██████████| 2028/2028 [04:05<00:00,  8.25it/s]

✅ Processed 10000 to 15000
🎉 DONE: 12028

Sentence: ['Ways', 'To', 'Politely', 'Turn', 'Down', 'a', 'Wedding', 'Invitation', '.', 'Buzz60', "'s", 'Elizabeth', 'Keatinge', 'shares', 'some', 'advice', 'on', 'how', 'to', 'politely', 'turn', 'down', 'a', 'wedding', 'invitation', '.']
Tags: ['O', 'O', 'O', 'O', 'O', 'O', 'B-MISC', 'O', 'O', 'B-PER', 'O', 'B-PER', 'I-PER', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-MISC', 'O', 'O']

Sentence: ['Michelle', 'Obama', 'wo', "n't", 'comment', 'on', 'Biden', 'apology', ',', 'holds', 'off', 'on', 'endorsement', 'at', 'Essence', 'Festival', '.', 'Six', 'Democratic', 'presidential', 'candidates', 'took', 'the', 'stage', 'at', 'the', 'Essence', 'Festival', 'on', 'Saturday', ',', 'but', 'it', 'was', 'a', 'former', 'inhabitant', 'of', 'the', 'White', 'House', 'who', 'got', 'a', 'rock', 'star', "'s", 'welcome', '.']
Tags: ['B-PER', 'I-PER', 'O', 'O', 'O', 'O', 'B-ORG', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-MISC', 'I-MISC', 'O', 'B-MISC', 'B-MIS

In [ ]:
import pickle

with open("/content/drive/MyDrive/NER_TRANS/processed.pkl", "wb") as f:
    pickle.dump({"sentences": sentences, "labels": labels}, f)

print("✅ Saved!")

✅ Saved!


In [3]:
import pickle

with open("/content/drive/MyDrive/NER_TRANS/processed.pkl", "rb") as f:
    data = pickle.load(f)

sentences = data["sentences"]
labels = data["labels"]

print("✅ Loaded!")
print("Total sentences:", len(sentences))

✅ Loaded!
Total sentences: 12028


In [4]:
print(len(sentences), len(labels))

12028 12028


## Batching + Tokenization + Saving

In [5]:
# Convert labels to IDs

unique_labels = sorted({tag for sent in labels for tag in sent})
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}

NUM_LABELS = len(label2id)

print("Labels:", label2id)

Labels: {'B-DATE': 0, 'B-LOC': 1, 'B-MISC': 2, 'B-ORG': 3, 'B-PER': 4, 'I-DATE': 5, 'I-LOC': 6, 'I-MISC': 7, 'I-ORG': 8, 'I-PER': 9, 'O': 10}


In [6]:
from transformers import BertTokenizerFast
import pickle
from tqdm import tqdm

tokenizer = BertTokenizerFast.from_pretrained("bert-base-cased")

def align_labels(encodings, labels):
    aligned_labels = []

    for i, label in enumerate(labels):
        word_ids = encodings.word_ids(batch_index=i)

        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)

            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label[word_idx]])

            else:
                current_label = label[word_idx]

                if current_label.startswith("B-"):
                    current_label = current_label.replace("B-", "I-")

                label_ids.append(label2id[current_label])

            previous_word_idx = word_idx

        aligned_labels.append(label_ids)

    return aligned_labels

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
print("🚀 Tokenizing...")

encodings = tokenizer(
        sentences,
        is_split_into_words=True,
        padding=True,
        truncation=True,
        max_length=128
    )

aligned_labels = align_labels(encodings, labels)

🚀 Tokenizing...


In [9]:
import pickle

with open("/content/drive/MyDrive/NER_TRANS/bert_ready_data.pkl", "wb") as f:
    pickle.dump({
        "input_ids": encodings["input_ids"],
        "attention_mask": encodings["attention_mask"],
        "labels": aligned_labels,
        "label2id": label2id,
        "id2label": id2label
    }, f)

print("✅ Saved successfully!")

✅ Saved successfully!


## Dataset class

In [11]:
import torch

class NERDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key:(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

dataset = NERDataset(encodings, aligned_labels)

## Train-test split

In [12]:
from sklearn.model_selection import train_test_split
import numpy as np

idx = np.arange(len(dataset))

train_idx, val_idx = train_test_split(idx, test_size=0.1, random_state=42, shuffle=True)

train_dataset = torch.utils.data.Subset(dataset, train_idx)
val_dataset = torch.utils.data.Subset(dataset, val_idx)

# Model

In [13]:
from transformers import BertForTokenClassification

id2label = {v: k for k, v in label2id.items()}

model = BertForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

print(label2id)
print(id2label)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

{'B-DATE': 0, 'B-LOC': 1, 'B-MISC': 2, 'B-ORG': 3, 'B-PER': 4, 'I-DATE': 5, 'I-LOC': 6, 'I-MISC': 7, 'I-ORG': 8, 'I-PER': 9, 'O': 10}
{0: 'B-DATE', 1: 'B-LOC', 2: 'B-MISC', 3: 'B-ORG', 4: 'B-PER', 5: 'I-DATE', 6: 'I-LOC', 7: 'I-MISC', 8: 'I-ORG', 9: 'I-PER', 10: 'O'}


## Training

In [15]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./bert_ner_output",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    learning_rate=2e-5,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset = val_dataset
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.331648,0.255172
2,0.228552,0.244787


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2708, training_loss=0.28010019277011977, metrics={'train_runtime': 698.5846, 'train_samples_per_second': 30.991, 'train_steps_per_second': 3.876, 'total_flos': 1414383772684800.0, 'train_loss': 0.28010019277011977, 'epoch': 2.0})

In [17]:
trainer.save_model("/content/drive/MyDrive/NER_TRANS/NER_BERT_MODEL")
tokenizer.save_pretrained("/content/drive/MyDrive/NER_TRANS/NER_BERT_MODEL")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/NER_TRANS/NER_BERT_MODEL/tokenizer_config.json',
 '/content/drive/MyDrive/NER_TRANS/NER_BERT_MODEL/tokenizer.json')

## Correct evaluation

In [16]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=0b237709b130ba51c0e9fd3450bb1cdb3f47d398d0b0b82eb280e2ddc496657a
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [18]:
from seqeval.metrics import classification_report

predictions, labels, _ = trainer.predict(val_dataset)

import numpy as np

preds = np.argmax(predictions, axis=2)

true_labels = []
true_preds = []

for pred, lab in zip(preds, labels):
    curr_preds = []
    curr_labels = []

    for p, l in zip(pred, lab):
        if l != -100:
            curr_preds.append(id2label[p])
            curr_labels.append(id2label[l])

    true_preds.append(curr_preds)
    true_labels.append(curr_labels)

print(classification_report(true_labels, true_preds))

              precision    recall  f1-score   support

        DATE       0.88      0.92      0.90      2301
         LOC       0.79      0.82      0.80      1951
        MISC       0.69      0.74      0.72      3904
         ORG       0.57      0.67      0.61      2882
         PER       0.72      0.79      0.75      3386

   micro avg       0.71      0.78      0.74     14424
   macro avg       0.73      0.79      0.76     14424
weighted avg       0.72      0.78      0.75     14424



In [22]:
import pandas as pd

bert_results = [{
    "Model": "BERT",
    "Embedding": "bert-base-cased",
    "Precision": 0.71,
    "Recall": 0.78,
    "F1 Score": 0.74
}]

df_bert = pd.DataFrame(bert_results)

print(df_bert)


  Model        Embedding  Precision  Recall  F1 Score
0  BERT  bert-base-cased       0.71    0.78      0.74


In [23]:
df_bert.to_csv("/content/drive/MyDrive/NER_TRANS/bert_results.csv", index=False)

print("✅ BERT results saved!")

✅ BERT results saved!


In [24]:
dl_df = pd.read_csv("/content/drive/MyDrive/NER_DL/dl_results.csv")
bert_df = pd.read_csv("/content/drive/MyDrive/NER_TRANS/bert_results.csv")

final_df = pd.concat([dl_df, bert_df], ignore_index=True)

print(final_df)

         Model        Embedding  Precision    Recall  F1 Score
0       BiLSTM         Word2Vec   0.634703  0.519518  0.571363
1       BiLSTM            GloVe   0.678954  0.474252  0.558435
2  BiLSTM-Char         Word2Vec   0.667575  0.508721  0.577422
3  BiLSTM-Char            GloVe   0.689557  0.600498  0.641953
4         BERT  bert-base-cased   0.710000  0.780000  0.740000


In [26]:
final_df.to_csv("/content/drive/MyDrive/NER_TRANS/final_comparison.csv", index=False)